# Multi-HARM — Project Review Demo (Colab T4, ~1-1.5 h)

Reduced-size **real-data** run: 400 clean MS-MARCO + 400 injected (4 attack types x 5 goals x 20) = 800 samples on **Llama-3.1-8B-Instruct (4-bit nf4)**.
Same pipeline as the full 2,000-sample run — only `demo_env.sh` sizes differ.

**Before running:** `!git clone <your-repo-url> && %cd <repo>` (or unzip the uploaded code zip).
**Staging:** run cells top to bottom. The long one is cell 03 (~30-75 min); everything after it is seconds-to-minutes on cached signals. If a cell fails, see `DEMO_RUNBOOK.md`.

| cell | stage | time |
|---|---|---|
| 01 | env + model download | 10-20 min |
| 02 | dataset (800) | ~3 min |
| 03 | §2.0 gate + signal extraction | **30-75 min** |
| 04-08 | H*, 4.8 rows 1-2, specialists, meta | <2 min total |
| 09 | all tables + latency + span audit | ~5 min |
| 10 | figures + RESULTS.md | seconds |

In [ ]:
!pip install -q -r requirements.txt
print("deps installed")

In [ ]:
# STAGE 01 — env report + model download + shape smoke test  (10-20 min)
!source ./demo_env.sh && python 01_setup_and_validate.py

In [ ]:
# STAGE 02 — dataset: 800 samples, 60/20/20 stratified by (type, goal)  (~3 min)
!source ./demo_env.sh && python 02_build_dataset.py

In [ ]:
# STAGE 03 — §2.0 token-range validation gate + full signal extraction
# THE LONG CELL (30-75 min). Resumable: if the runtime dies, restart the same
# runtime and re-run this cell — it skips already-extracted samples.
!source ./demo_env.sh && python 03_extract_signals.py

In [ ]:
# STAGE 04 — pooled H* (per-head AUROC, 160 calibration samples)
!source ./demo_env.sh && python 04_calibrate_hstar.py

In [ ]:
# STAGE 05 — 4.8 ROW 1: attention-only, shared calibration (Attention Tracker replication)
!source ./demo_env.sh && python 05_baseline_attn_tracker.py

In [ ]:
# STAGE 06 — 4.8 ROW 2: HARM_general (fused, shared) + PIShield-style hidden-only baseline
!source ./demo_env.sh && python 06_calibrate_general.py

In [ ]:
# STAGE 07 — 4 type specialists: L* -> probe -> h_base -> alpha -> theta
# prints the half-split AUROC table (Phase 3 addition)
!source ./demo_env.sh && python 07_calibrate_specialists.py

In [ ]:
# STAGE 08 — meta-decision layer evaluated on VAL (FPR criterion check)
!source ./demo_env.sh && python 08_meta_decision.py

In [ ]:
# STAGE 09 — Tables A-E, 4.3/4.4, 4.8 spine table, span-width audit, latency
# --with-model adds the forward-included latency overhead (loads model once)
!source ./demo_env.sh && python 09_experiments_analysis.py --with-model

In [ ]:
# STAGE 10 — figures + paper-facing report
!source ./demo_env.sh && python 10_figures_report.py
print()
print("=== DEMO READY ===")
print("  out/experiments/SUMMARY.md    <- criteria check + 4.8 table")
print("  out/report/RESULTS.md         <- 4.8 lead table, 4.9 BAGEL/Luna-2 table")
print("  out/figures/*.png             <- all figures")